# 6.1 Audio Encoder — Design Experiment (Step 1 of 2)

**Tujuan:** memilih **audio encoder terbaik dan paling cocok** untuk *text-guided automatic gain staging*
pada MUSDB18-HQ, dengan perbandingan **apple-to-apple**.

**Posisi dalam eksperimen global:**
1. **(Notebook ini)** Cari audio encoder terbaik & paling cocok.
2. Seluruh jenis *fusion* dicoba memakai audio encoder pemenang untuk mencari kombinasi terbaik.

---

## Kontrak fairness (dikunci)

| Aspek | Nilai (sama untuk semua run) |
| --- | --- |
| Text encoder | `indobenchmark/indobert-base-p1` — **di-fine-tune, tidak frozen** |
| Head / arsitektur | **M3 Structured Two-Branch**: audio → `L_pred` (4), text → `r_pred` (4), `g = L_ref + r - L` |
| Loss | `MSE(g) + 0.1*MSE(L) + 0.1*MSE(r)` (aux dimatikan sesuai mode ablasi) |
| Label | rumus yang sudah ada di `wadidaw/data/features.py` (`compute_gain_targets`) |
| Data / split / prompt | manifest yang sudah ada, tidak diubah |
| Optimizer | AdamW, encoder LR `1e-4`, head LR `1e-3`, cosine, grad clip `1.0`, AMP |
| Seeds | `[42, 123, 2026]` |
| Metrik | `wadidaw.training.evaluate.evaluate_model` (MAE/RMSE/band/success/adherence/per-stem) |

**Yang berubah hanya audio encoder:** `cnn_scratch`, `ast`, `mert`, `beats` — semuanya **di-fine-tune (tidak ada yang frozen)**.

> Catatan: "seluruh parameter sama" berarti **protokol/hyperparameter/seed sama**, bukan jumlah parameter
> identik (encoder 12M vs 95M memang tak bisa disamakan jumlah parameternya). LR encoder dipisah
> (`1e-4`) dari LR head (`1e-3`) persis seperti bagian 7.1 dokumentasi.

## Ablasi unimodal & bimodal

M3 memecah prediksi menjadi dua cabang, sehingga tiga mode bisa dijalankan dari head yang sama:
- `bimodal` — audio + teks: `g = L_ref + r - L` (late fusion).
- `audio_only` — teks dimatikan (`r = 0`): `g = L_ref - L` (equal-loudness).
- `text_only` — audio dimatikan (`L = 0`): `g = r`.

## Fitur level eksplisit (wajib untuk AST/MERT/BEATs)

AST/MERT/BEATs menghasilkan embedding yang **gain-invariant** (tidak menyimpan loudness absolut `g0`).
Karena itu tiap stem diberi fitur level eksplisit: **RMS dBFS, energi band low/mid/high, peak dB,
crest factor, active mask**. Fitur ini dihitung dari **input** (bukan label) lalu disuntikkan ke token:
`token = proj(encoder) + stem_embedding + proj_level(features)`.

## Output
- `outputs/encoder_selection/audio_encoder_results.csv` (+ per-run JSON & figure).
- Pemenang → dipakai di step 2 (sweep fusion).

## Cara menjalankan (per anggota)

1. `git pull`, lalu set `MEMBER = <nomor>` di **Cell 1** (tabel pembagian tepat di bawah Cell 1).
2. Jalankan sel berurutan. Cell 2 membangun cache **waveform 16 kHz** (`data/cache/waveform16k/`)
   dari stem MUSDB 44.1 kHz — hanya perlu untuk anggota `ast`/`mert`/`beats` (~1.5 GB, sekali saja).
3. Set `RUN_EXPERIMENT = True` pada sel *Run matrix* untuk melatih shard milikmu. Setiap run
   tersimpan ke `outputs/encoder_selection/member<N>/`.
4. Commit `outputs/encoder_selection/member<N>/*.json` lalu push. Ketua `git pull` dan menjalankan
   sel *Leaderboard* untuk menggabungkan semua anggota.

**Dependensi tambahan** (di luar `requirements.txt`):
- `transformers` (sudah ada) untuk AST + IndoBERT.
- MERT: `trust_remote_code=True` (kode remote dari `m-a-p/MERT-v1-95M`).
- BEATs: paket `beats` dari Microsoft unilm
  (`pip install git+https://github.com/microsoft/unilm.git#subdirectory=beats`) + checkpoint
  `BEATs_iter3_plus_AS2M.pt` (diunduh otomatis dari mirror HF `lpepino/beats_ckpts`).
- `matplotlib` untuk figure.

In [ ]:
# ----------------------------------------------------------------------------- Cell 1: bootstrap + config
import copy
import json
import random
import sys
import time
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "wadidaw").is_dir():
            return candidate
    return start


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from wadidaw.data.dataset import SAMPLES_ROOT, MusdbMultimodalDataset
from wadidaw.data.features import ACTIVE_THRESHOLD_DB, resample_to_16k_mono
from wadidaw.data.musdb import MUSDB_ROOT
from wadidaw.data.prompts import STEM_NAMES
from wadidaw.models.structured import compute_multimodal_loss
from wadidaw.training.evaluate import evaluate_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WAVEFORM_CACHE = ROOT / "data" / "cache" / "waveform16k"
OUT_DIR = ROOT / "outputs" / "encoder_selection"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "train_manifest": str(SAMPLES_ROOT / "train_samples.json"),
    "val_manifest": str(SAMPLES_ROOT / "validation_samples.json"),
    "test_human_manifest": str(SAMPLES_ROOT / "test_human_samples.json"),
    "test_synthetic_manifest": str(SAMPLES_ROOT / "test_synthetic_samples.json"),
    "text_model": "indobenchmark/indobert-base-p1",
    "d_model": 128,
    "dropout": 0.15,
    "text_hidden": 128,
    "batch_size": 4,          # dikunci 4 di semua mesin
    "grad_accum_steps": 4,    # dikunci 4 di semua mesin -> effective batch 16
    "epochs": 30,
    "patience": 5,
    "lr_head": 1e-3,
    "lr_encoder": 1e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,
    "seeds": [42, 123, 2026],
    "use_amp": True,
    "encoders": ["cnn_scratch", "ast", "mert", "beats"],
    "modes": ["bimodal", "audio_only", "text_only"],
    "waveform_rate": 16000,
}
# Auxiliary-loss weights per ablation mode (lambda_loudness, lambda_balance).
AUX_LAMBDA = {"bimodal": (0.1, 0.1), "audio_only": (0.1, 0.0), "text_only": (0.0, 0.1)}

# --- Pembagian run matrix ke 4 anggota (sesuai spek mesin dokumentasi 10.1) -----
# Setiap anggota mengubah MEMBER ke nomor mesinnya lalu menjalankan notebook ini.
# text_only TIDAK memakai audio encoder (g = r), jadi cukup dijalankan sekali (member 2).
MEMBER = 1
SHARDS = {
    1: {"encoders": ["cnn_scratch"], "modes": ["bimodal", "audio_only"]},                 # iGPU Arc / CPU, 16 GB
    2: {"encoders": ["ast"],          "modes": ["bimodal", "audio_only", "text_only"]},   # RTX 3050 4 GB, 24 GB
    3: {"encoders": ["mert"],         "modes": ["bimodal", "audio_only"]},                 # RTX 4050 6 GB, 32 GB
    4: {"encoders": ["beats"],        "modes": ["bimodal", "audio_only"]},                 # RTX 5050 8 GB, 16 GB
}
effective_batch = 16  # batch_size * grad_accum_steps, sama di semua mesin (apple-to-apple)
assert CONFIG["batch_size"] * CONFIG["grad_accum_steps"] == effective_batch, "effective batch harus 16"
assert MEMBER in SHARDS, f"MEMBER harus 1..4, bukan {MEMBER}"
RUN_DIR = OUT_DIR / f"member{MEMBER}"

print("device:", DEVICE)
print("repo root:", ROOT)
print("waveform cache:", WAVEFORM_CACHE)
print(f"member {MEMBER}: encoders={SHARDS[MEMBER]['encoders']} modes={SHARDS[MEMBER]['modes']} "
      f"(effective batch {effective_batch})")

## Pembagian run matrix ke 4 anggota

Matriks unik = **27 run** (bukan 36): mode `text_only` tidak memakai audio encoder (`g = r`),
jadi cukup dijalankan **sekali** oleh satu anggota dan hasilnya sama untuk semua encoder.

| Anggota | Mesin (dokumentasi §10.1) | Encoder | Mode | Seed | Run |
| --- | --- | --- | --- | --- | --- |
| 1 | iGPU Intel Arc, **tanpa CUDA**, 16 GB | `cnn_scratch` | bimodal, audio_only | 42/123/2026 | 6 |
| 2 | RTX 3050 **4 GB**, 24 GB | `ast` | bimodal, audio_only | 42/123/2026 | 6 |
| 2 | RTX 3050 **4 GB** | — (`text_only` encoder-independent) | text_only | 42/123/2026 | 3 |
| 3 | RTX 4050 **6 GB**, 32 GB | `mert` | bimodal, audio_only | 42/123/2026 | 6 |
| 4 | RTX 5050 **8 GB**, 16 GB | `beats` | bimodal, audio_only | 42/123/2026 | 6 |

Batch dikunci `4 × 4 = effective 16` di semua mesin agar perbandingan apple-to-apple.

### Cara jalan tiap anggota
1. `git pull`.
2. Set `MEMBER = <nomor>` di **Cell 1**.
3. Jalankan Cell 1–9. Cache waveform 16 kHz dibuat otomatis bila belum ada
   (hanya perlu untuk anggota `ast`/`mert`/`beats`).
4. Set `RUN_EXPERIMENT = True` di **Cell 9 (Run matrix)**. Setiap run disimpan langsung ke
   `outputs/encoder_selection/member<N>/` sehingga progres selamat bila dihentikan di tengah.
5. Commit `outputs/encoder_selection/member<N>/*.json` lalu push (`outputs/` tidak di-ignore git).

### Cara merge (ketua)
Setelah semua anggota push: `git pull`, lalu jalankan **Cell 10 (Leaderboard)** — sel itu membaca
semua `outputs/encoder_selection/member*/*.json` dari disk dan menulis `audio_encoder_results.csv`
gabungan + leaderboard.

> Anggota 2 (4 GB): AST fine-tune `batch 4` bisa OOM. Fallback yang tetap adil adalah
> `shortcut: turunkan batch_size dan naikkan grad_accum_steps agar batch_size*grad_accum_steps tetap 16`
> (mis. `2×8`), lalu catat di laporan. Jangan turunkan effective batch.

## 1. Cache waveform 16 kHz

AST/BEATs memakai 16 kHz, MERT-v1 aslinya 24 kHz. Kita cache **satu** waveform 16 kHz mono
(reuse `resample_to_16k_mono`, konsisten dengan cache log-mel) dan memberi MERT input 16 kHz
sebagai *documented domain shift*; upgrade ke 24 kHz jika MERT menang.

Setiap excerpt di-cache sebagai `(4, 128000)` float16 (stem urut `STEM_NAMES`), **raw tanpa g0** —
perturbasi g0 diterapkan saat load, sama seperti cache log-mel.

In [2]:
# ----------------------------------------------------------------------------- Cell 2: waveform cache (idempotent)


def excerpt_id_from_audio_path(audio_path: str) -> tuple[str, str]:
    p = Path(audio_path)
    return p.stem, p.parent.name  # (excerpt_id, split_tag)


def waveform_cache_path(audio_path: str) -> Path:
    ex_id, split_tag = excerpt_id_from_audio_path(audio_path)
    return WAVEFORM_CACHE / split_tag / f"{ex_id}.npy"


def musdb_subset_for_split(split_tag: str) -> str:
    return "train" if split_tag in ("train", "validation") else "test"


def build_waveform_cache(manifests, overwrite: bool = False) -> None:
    """Cache 16 kHz mono (4, 128000) float16 waveforms for every excerpt in the manifests."""
    unique: dict[str, dict] = {}
    for manifest in manifests:
        if not Path(manifest).is_file():
            continue
        for record in MusdbMultimodalDataset(manifest).samples:
            unique.setdefault(str(record["audio_path"]), record)

    built = 0
    for audio_path, record in unique.items():
        npy_path = waveform_cache_path(audio_path)
        if npy_path.exists() and not overwrite:
            continue
        _ex_id, split_tag = excerpt_id_from_audio_path(audio_path)
        song_dir = MUSDB_ROOT / musdb_subset_for_split(split_tag) / str(record["song"])
        start = int(round(float(record["excerpt_start_sec"]) * 44100))
        frames = int(round(float(record["duration_sec"]) * 44100))
        stems = []
        for stem in STEM_NAMES:
            audio, sr = sf.read(song_dir / f"{stem}.wav", start=start, frames=frames, dtype="float32")
            stems.append(resample_to_16k_mono(audio, orig_sr=sr))
        npy_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(npy_path, np.stack(stems, axis=0).astype(np.float16))
        built += 1
    print(f"waveform cache: built {built}, total excerpts {len(unique)}")


CACHE_MANIFESTS = [CONFIG[k] for k in ("train_manifest", "val_manifest", "test_human_manifest", "test_synthetic_manifest")]
if not WAVEFORM_CACHE.exists():
    build_waveform_cache(CACHE_MANIFESTS)
else:
    print("waveform cache already present:", WAVEFORM_CACHE)

waveform cache: built 1500, total excerpts 1500


## 2. Dataset

`EncoderDataset` membungkus `MusdbMultimodalDataset` (log-mel `(4, 64, 800)` + label) dan menambahkan
`waveform` `(4, 128000)` dengan perturbasi `g0` yang sama. `cache_in_memory=False` untuk menekan RAM.

In [3]:
# ----------------------------------------------------------------------------- Cell 3: dataset


class EncoderDataset(Dataset):
    """MusdbMultimodalDataset + raw 16 kHz waveform (same g0 perturbation as the log-mel input)."""

    def __init__(self, manifest, root_dir=ROOT, cache_in_memory: bool = False):
        self.root_dir = Path(root_dir)
        self.base = MusdbMultimodalDataset(manifest, root_dir=root_dir, cache_in_memory=cache_in_memory)

    def __len__(self) -> int:
        return len(self.base)

    def __getitem__(self, index: int) -> dict:
        item = self.base[index]
        record = self.base.samples[index]
        wf_path = waveform_cache_path(str(record["audio_path"]))
        if not wf_path.is_file():
            raise FileNotFoundError(f"Missing waveform cache {wf_path}. Run Cell 2 first.")
        waveform = np.load(wf_path).astype(np.float32)
        g0 = np.asarray(record["g0_db"], dtype=np.float32)
        mask = np.asarray(record.get("active_mask", [True] * 4), dtype=bool)
        eff = np.where(mask, g0, 0.0).astype(np.float32)
        item["waveform"] = torch.from_numpy(waveform * np.power(10.0, eff / 20.0)[:, None])
        return item


_ds = EncoderDataset(CONFIG["val_manifest"])
_s = _ds[0]
print("keys:", [k for k in _s if k in ("audio", "waveform", "gains_db", "loudness_db", "balance_target", "active_mask")])
print("audio", tuple(_s["audio"].shape), "| waveform", tuple(_s["waveform"].shape), "| gains", tuple(_s["gains_db"].shape))

keys: ['audio', 'loudness_db', 'balance_target', 'gains_db', 'active_mask', 'waveform']
audio (4, 64, 800) | waveform (4, 128000) | gains (4,)


## 3. Fitur level eksplisit

Dihitung **dari input** encoder (bukan dari label), jadi tidak ada leakage:
`N_LEVEL_FEATS = 7` → RMS dBFS, band-low, band-mid, band-high (dB), peak dB, crest factor, active mask.

In [4]:
# ----------------------------------------------------------------------------- Cell 4: level features
N_LEVEL_FEATS = 7  # rms, band_low, band_mid, band_high, peak, crest, active


def _assemble_level(rms_db: torch.Tensor, bands: torch.Tensor, peak_db: torch.Tensor) -> torch.Tensor:
    crest = peak_db - rms_db
    active = (rms_db > ACTIVE_THRESHOLD_DB).to(rms_db.dtype)
    stacked = torch.stack(
        [rms_db, bands[..., 0], bands[..., 1], bands[..., 2], peak_db, crest, active * 30.0], dim=-1
    )
    return stacked / 30.0  # rough fixed scaling; the level MLP handles the rest


def mel_level_features(audio_norm: torch.Tensor) -> torch.Tensor:
    """Level features from normalized dBFS log-mel (B, 4, 64, T). dB = x * 50 - 30."""
    db = audio_norm * 50.0 - 30.0
    power = torch.pow(10.0, db / 10.0)
    rms_db = 10.0 * torch.log10(power.mean(dim=(2, 3)).clamp_min(1e-12))
    peak_db = db.amax(dim=(2, 3))
    bands = 10.0 * torch.log10(
        torch.stack(
            [power[:, :, :16].mean(dim=(2, 3)), power[:, :, 16:40].mean(dim=(2, 3)), power[:, :, 40:].mean(dim=(2, 3))],
            dim=-1,
        ).clamp_min(1e-12)
    )
    return _assemble_level(rms_db, bands, peak_db)


def waveform_level_features(waveform: torch.Tensor, sample_rate: int = 16000) -> torch.Tensor:
    """Level features from raw waveform (B, 4, S)."""
    x = waveform.float()
    rms_db = 20.0 * torch.log10(x.pow(2).mean(dim=-1).clamp_min(1e-12).sqrt())
    peak_db = 20.0 * torch.log10(x.abs().amax(dim=-1).clamp_min(1e-12))
    spec = torch.fft.rfft(x, dim=-1)
    power = spec.real.pow(2) + spec.imag.pow(2)
    freqs = torch.fft.rfftfreq(x.shape[-1], d=1.0 / sample_rate).to(x.device)
    band_db = []
    for lo, hi in ((0, 250), (250, 2000), (2000, sample_rate // 2)):
        sel = (freqs >= lo) & (freqs < hi)
        band_db.append(10.0 * torch.log10(power[..., sel].mean(dim=-1).clamp_min(1e-12)))
    return _assemble_level(rms_db, torch.stack(band_db, dim=-1), peak_db)


def compute_level_features(x: torch.Tensor, input_key: str) -> torch.Tensor:
    """All encoders use the same waveform-level features.

    The encoder content path may differ, but level features are always
    computed from the 16 kHz waveform so RMS/band/peak values are directly
    comparable in dBFS.
    """
    if input_key != "waveform":
        raise ValueError(f"Level features require waveform input, got {input_key!r}.")
    return waveform_level_features(x)

## 4. Text encoder yang di-fine-tune

`TextEncoderWrapper` bawaan proyek membekukan encoder teks (`@torch.no_grad` + cache), sehingga
tidak bisa dipakai untuk fine-tune. Cabang kecil di bawah ini **meneruskan gradien** ke IndoBERT.

In [5]:
# ----------------------------------------------------------------------------- Cell 5: trainable text branch


class TrainableIndoBertTextBranch(nn.Module):
    """Fine-tunable IndoBERT producing bounded balance targets r (B, 4)."""

    is_pretrained = True

    def __init__(self, model_name: str, hidden_dim: int = 128, dropout: float = 0.15, max_length: int = 64):
        super().__init__()
        from transformers import AutoModel, AutoTokenizer

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.max_length = max_length
        embed_dim = int(self.model.config.hidden_size)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 4),
        )

    def forward(self, prompts, device) -> torch.Tensor:
        encoded = self.tokenizer(
            list(prompts), padding=True, truncation=True, max_length=self.max_length, return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        hidden = self.model(**encoded).last_hidden_state
        mask = encoded["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1.0)
        return 6.0 * torch.tanh(self.head(pooled) / 6.0)

## 5. Audio encoder registry

Antarmuka seragam: `forward(input) -> tokens (B, 4, T', D)`.
- `cnn_scratch` → CNN 1D scratch pada waveform 16 kHz, menghasilkan 25 token temporal.
- `ast` / `mert` / `beats` → encoder pretrained dari waveform (S = 128000), dijalankan per-stem
  dengan bobot yang sama (shared), semua parameter **trainable**.
- Semua encoder memakai input waveform 16 kHz dan level features dari waveform yang sama.

In [6]:
# ----------------------------------------------------------------------------- Cell 6: audio encoders
BEATS_URL = "https://huggingface.co/lpepino/beats_ckpts/resolve/main/BEATs_iter3_plus_AS2M.pt"


def download_beats_checkpoint(path: Path) -> Path:
    path = Path(path)
    if path.is_file():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"downloading BEATs checkpoint -> {path}")
    from urllib.request import urlretrieve

    urlretrieve(BEATS_URL, path)
    return path


class ScratchWaveformEncoder(nn.Module):
    """Scratch 1D CNN on raw 16 kHz waveform, returning 25 temporal tokens per stem."""

    input_key = "waveform"
    is_pretrained = False

    def __init__(self, embed_dim: int = 128, n_tokens: int = 25):
        super().__init__()
        self.embed_dim = embed_dim
        self.n_tokens = n_tokens
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, stride=4, padding=3),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Conv1d(32, 64, kernel_size=5, stride=4, padding=2),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Conv1d(64, 128, kernel_size=3, stride=4, padding=1),
            nn.BatchNorm1d(128),
            nn.GELU(),
        )
        self.proj = nn.Sequential(
            nn.Linear(128, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
        )

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        bsz, n_stems, n_samples = waveform.shape
        x = waveform.reshape(bsz * n_stems, 1, n_samples)
        h = self.conv(x)
        h = F.adaptive_avg_pool1d(h, self.n_tokens)
        h = h.transpose(1, 2)
        h = self.proj(h)
        return h.reshape(bsz, n_stems, self.n_tokens, self.embed_dim)


class _HFWaveformEncoder(nn.Module):
    input_key = "waveform"
    is_pretrained = True
    sample_rate = 16000

    def _encode_waveform(self, waveforms: torch.Tensor) -> torch.Tensor:  # (N, S) -> (N, T, D)
        raise NotImplementedError

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        bsz, n_stems, _ = waveform.shape
        flat = waveform.reshape(bsz * n_stems, waveform.shape[-1])
        hidden = self._encode_waveform(flat)
        return hidden.reshape(bsz, n_stems, hidden.shape[-2], hidden.shape[-1])


class ASTEncoder(_HFWaveformEncoder):
    def __init__(self, model_name: str = "MIT/ast-finetuned-audioset-10-10-0.4593"):
        super().__init__()
        from transformers import ASTFeatureExtractor, ASTModel

        self.extractor = ASTFeatureExtractor.from_pretrained(model_name)
        self.model = ASTModel.from_pretrained(model_name)
        self.embed_dim = int(self.model.config.hidden_size)

    def _encode_waveform(self, waveforms: torch.Tensor) -> torch.Tensor:
        feats = self.extractor(
            list(waveforms.detach().cpu().numpy()), sampling_rate=self.sample_rate, return_tensors="pt"
        )
        return self.model(feats["input_values"].to(waveforms.device)).last_hidden_state


class MERTEncoder(_HFWaveformEncoder):
    def __init__(self, model_name: str = "m-a-p/MERT-v1-95M"):
        super().__init__()
        from transformers import AutoModel, Wav2Vec2FeatureExtractor

        self.model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        self.extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name, trust_remote_code=True)
        self.embed_dim = int(self.model.config.hidden_size)

    def _encode_waveform(self, waveforms: torch.Tensor) -> torch.Tensor:
        feats = self.extractor(
            list(waveforms.detach().cpu().numpy()), sampling_rate=self.sample_rate, return_tensors="pt"
        )
        outputs = self.model(feats["input_values"].to(waveforms.device))
        return outputs.last_hidden_state


class BEATsEncoder(_HFWaveformEncoder):
    def __init__(self, checkpoint: str | None = None):
        super().__init__()
        try:
            from beats.BEATs import BEATs, BEATsConfig
        except Exception as exc:  # pragma: no cover - dependency gate
            raise RuntimeError(
                "BEATs requires Microsoft's `beats` package: "
                "`pip install git+https://github.com/microsoft/unilm.git#subdirectory=beats`. "
                f"Original error: {exc}"
            ) from exc

        ckpt = download_beats_checkpoint(Path(checkpoint) if checkpoint else Path.home() / "beats_data" / "BEATs_iter3_plus_AS2M.pt")
        payload = torch.load(ckpt, map_location="cpu")
        config = BEATsConfig(payload["cfg"])
        self.model = BEATs(config)
        self.model.load_state_dict(payload["model"])
        self.embed_dim = int(config.encoder_embed_dim)

    def _encode_waveform(self, waveforms: torch.Tensor) -> torch.Tensor:
        out = self.model.extract_features(waveforms)
        return out[0] if isinstance(out, tuple) else out


ENCODER_REGISTRY = {
    "cnn_scratch": ScratchWaveformEncoder,
    "ast": ASTEncoder,
    "mert": MERTEncoder,
    "beats": BEATsEncoder,
}


def build_encoder(name: str) -> nn.Module:
    if name not in ENCODER_REGISTRY:
        raise ValueError(f"Unknown encoder '{name}'. Available: {sorted(ENCODER_REGISTRY)}")
    return ENCODER_REGISTRY[name]()

## 6. `EncoderProbeModel` (audio encoder + IndoBERT + head M3)

Hanya blok audio encoder yang berganti. Head M3, rumus `g`, dan kontrak loss tetap identik.
`audio_input_key` sekarang selalu `waveform`, sehingga semua encoder dievaluasi dengan
`evaluate_model` yang sama.

In [ ]:
# ----------------------------------------------------------------------------- Cell 7: model


class EncoderProbeModel(nn.Module):
    """M3 structured two-branch head with a swappable audio encoder and a fine-tuned IndoBERT branch."""

    def __init__(self, encoder_name: str, mode: str, config: dict):
        super().__init__()
        if mode not in ("bimodal", "audio_only", "text_only"):
            raise ValueError(f"Invalid mode: {mode}")
        self.mode = mode
        self.encoder_name = encoder_name
        # text_only tidak memakai audio encoder (g = r): jangan bangun/unduh encoder berat.
        self.encoder = None if mode == "text_only" else build_encoder(encoder_name)
        self.audio_input_key = self.encoder.input_key if self.encoder is not None else "audio"

        d = config["d_model"]
        self.enc_proj = nn.Linear(self.encoder.embed_dim if self.encoder is not None else d, d)
        self.stem_embedding = nn.Parameter(torch.zeros(1, 4, 1, d))
        self.level_mlp = nn.Sequential(nn.Linear(N_LEVEL_FEATS, d), nn.GELU(), nn.Linear(d, d))
        self.stem_norm = nn.LayerNorm(d)
        self.loudness_head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, 1))
        self.text_branch = TrainableIndoBertTextBranch(config["text_model"], config["text_hidden"], config["dropout"])

    def forward(self, audio_input, prompts, active_mask=None, clamp_output: bool = False) -> dict:
        device = audio_input.device
        bsz = audio_input.shape[0]
        if active_mask is not None:
            mask = active_mask.to(device=device, dtype=torch.float32)
        else:
            mask = torch.ones((bsz, 4), device=device)
        count = mask.sum(dim=1, keepdim=True).clamp_min(1.0)

        l_pred = torch.zeros((bsz, 4), device=device)
        if self.mode != "text_only":
            level = compute_level_features(audio_input, self.audio_input_key)
            tokens = self.encoder(audio_input)
            h = self.enc_proj(tokens) + self.stem_embedding + self.level_mlp(level).unsqueeze(2)
            l_pred = self.loudness_head(self.stem_norm(h.mean(dim=2))).squeeze(-1)
        l_ref = (l_pred * mask).sum(dim=1, keepdim=True) / count

        r_pred = torch.zeros((bsz, 4), device=device)
        if self.mode != "audio_only":
            raw_r = self.text_branch(prompts, device)
            r_pred = (raw_r - (raw_r * mask).sum(dim=1, keepdim=True) / count) * mask

        g_pred = (l_ref + r_pred - l_pred) * mask
        if clamp_output or not self.training:
            g_pred = g_pred.clamp(-18.0, 12.0)
        return {"gains_pred": g_pred, "loudness_pred": l_pred, "balance_pred": r_pred}

    def build_param_groups(self, config: dict) -> list[dict]:
        groups: list[dict] = []
        if self.encoder is not None:
            enc_lr = config["lr_encoder"] if getattr(self.encoder, "is_pretrained", False) else config["lr_head"]
            groups.append({"params": list(self.encoder.parameters()), "lr": enc_lr})
        head_modules = [self.enc_proj, self.level_mlp, self.stem_norm, self.loudness_head, self.text_branch.head]
        head_params = [p for module in head_modules for p in module.parameters()] + [self.stem_embedding]
        groups.append({"params": list(self.text_branch.model.parameters()), "lr": config["lr_encoder"]})
        groups.append({"params": head_params, "lr": config["lr_head"]})
        return groups

## 7. Train / eval loop

Reuse `compute_multimodal_loss` dan `evaluate_model`. Early stopping pada val MAE, AMP, gradient
accumulation, cosine schedule. Bobot aux loss mengikuti mode ablasi.

In [ ]:
# ----------------------------------------------------------------------------- Cell 8: training loop


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def env_info() -> dict:
    info = {"torch": torch.__version__, "device": str(DEVICE)}
    if torch.cuda.is_available():
        info["gpu"] = torch.cuda.get_device_name(0)
        info["cuda"] = torch.version.cuda
    return info


def run_one(encoder_name: str, mode: str, seed: int, config: dict, train_ds: Dataset, val_ds: Dataset, device=DEVICE) -> dict:
    seed_everything(seed)
    model = EncoderProbeModel(encoder_name, mode, config).to(device)
    groups = model.build_param_groups(config)
    optimizer = torch.optim.AdamW(groups, weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(config["epochs"], 1), eta_min=config["lr_encoder"] * 0.05
    )
    amp = bool(config["use_amp"] and device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    all_params = [p for g in groups for p in g["params"]]

    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False, num_workers=0)
    accum = max(1, int(config["grad_accum_steps"]))
    lam_l, lam_r = AUX_LAMBDA[mode]

    best = {"val_mae": float("inf"), "epoch": 0, "state": None, "metrics": {}}
    history: list[dict] = []
    t0 = time.time()
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    for epoch in range(1, config["epochs"] + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(train_loader):
            audio = batch[model.audio_input_key].to(device=device, dtype=torch.float32)
            mask = batch["active_mask"].to(device=device)
            with torch.amp.autocast(device_type=device.type, enabled=amp):
                out = model(audio, batch["prompt"], active_mask=mask, clamp_output=False)
                losses = compute_multimodal_loss(
                    out,
                    batch["gains_db"].to(device=device, dtype=torch.float32),
                    batch["loudness_db"].to(device=device, dtype=torch.float32),
                    batch["balance_target"].to(device=device, dtype=torch.float32),
                    active_mask=mask,
                    lambda_loudness=lam_l,
                    lambda_balance=lam_r,
                )
                loss = losses["loss"]
            scaler.scale(loss / accum).backward()
            if (step + 1) % accum == 0 or (step + 1) == len(train_loader):
                if config["grad_clip"] > 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(all_params, config["grad_clip"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        scheduler.step()

        val = evaluate_model(model, val_loader, device=device)
        history.append({
            "epoch": epoch,
            "val_mae_db": val["mae_db"],
            "val_rmse_db": val["rmse_db"],
            "val_band_mae_db": val["band_mae_db"],
            "val_success_1db": val["success_rate_1db"],
        })
        print(
            f"[{encoder_name:11s}/{mode:10s}/seed{seed}] ep{epoch:02d} "
            f"val_MAE={val['mae_db']:.4f} RMSE={val['rmse_db']:.4f} band={val['band_mae_db']:.4f}"
        )
        if val["mae_db"] < best["val_mae"] - 1e-4:
            best = {"val_mae": val["mae_db"], "epoch": epoch, "state": copy.deepcopy(model.state_dict()), "metrics": val}
        elif config["patience"] and (epoch - best["epoch"]) >= config["patience"]:
            print(f"  early stop at epoch {epoch} (best {best['epoch']})")
            break

    if best["state"] is not None:
        model.load_state_dict(best["state"])

    test_human = None
    if Path(config["test_human_manifest"]).is_file() and WAVEFORM_CACHE.exists():
        test_loader = DataLoader(EncoderDataset(config["test_human_manifest"]), batch_size=config["batch_size"], shuffle=False)
        test_human = evaluate_model(model, test_loader, device=device)

    return {
        "encoder": encoder_name,
        "mode": mode,
        "seed": seed,
        "member": MEMBER,
        "env": env_info(),
        "trainable_params": n_params,
        "train_sec": round(time.time() - t0, 1),
        "best_epoch": best["epoch"],
        "validation": best["metrics"],
        "test_human": test_human,
        "history": history,
    }

## 8. Run matrix (shard per anggota)

Matriks unik **27 run** dibagi ke 4 anggota: `4 encoder × {bimodal, audio_only} × 3 seed` plus
`text_only × 3 seed` (dijalankan satu kali karena tidak memakai encoder). Cell 9 hanya menjalankan
shard `MEMBER`, menulisnya ke `outputs/encoder_selection/member<N>/`, dan tidak dijalankan otomatis
(set `RUN_EXPERIMENT = True`).

In [ ]:
# ----------------------------------------------------------------------------- Cell 9: run matrix (shard anggota)
RUN_EXPERIMENT = False   # <-- set True untuk benar-benar melatih

RUNS: list[dict] = []
if RUN_EXPERIMENT:
    shard = SHARDS[MEMBER]
    train_ds = EncoderDataset(CONFIG["train_manifest"])
    val_ds = EncoderDataset(CONFIG["val_manifest"])
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    for encoder_name in shard["encoders"]:
        for mode in shard["modes"]:
            for seed in CONFIG["seeds"]:
                run = run_one(encoder_name, mode, seed, CONFIG, train_ds, val_ds)
                RUNS.append(run)
                # Simpan per-run langsung supaya progres selamat bila dihentikan di tengah.
                run_path = RUN_DIR / f"{encoder_name}_{mode}_seed{seed}.json"
                run_path.write_text(json.dumps(run, indent=2, default=str))
                print("saved", run_path)
    print(f"member {MEMBER}: saved {len(RUNS)} per-run JSON -> {RUN_DIR}")
else:
    print(f"RUN_EXPERIMENT is False; member {MEMBER} shard skipped.")

## 9. Leaderboard & figure

Pemenang "terbaik" = val MAE terendah (mean ± std antar seed). "Paling cocok" = akurasi vs biaya
(jumlah parameter & waktu latih).

In [ ]:
# ----------------------------------------------------------------------------- Cell 10: aggregate + plot
import collections


def load_all_runs() -> tuple[list[dict], list[str]]:
    """Read every per-member JSON from disk (merge across the 4 machines), fall back to session RUNS."""
    runs, sources = [], []
    for path in sorted(OUT_DIR.glob("member*/*.json")):
        try:
            runs.append(json.loads(path.read_text()))
            sources.append(path.parent.name)
        except Exception as exc:  # noqa: BLE001
            print("skip", path, exc)
    if not runs:
        return list(globals().get("RUNS", [])), sorted({str(r.get("member")) for r in globals().get("RUNS", [])})
    return runs, sorted(set(sources))


def aggregate(runs: list[dict]) -> list[dict]:
    grouped = collections.defaultdict(list)
    for run in runs:
        grouped[(run["encoder"], run["mode"])].append(run)
    rows = []
    for (encoder, mode), group in sorted(grouped.items()):
        def mean_std(key):
            vals = [float(g["validation"][key]) for g in group]
            return round(float(np.mean(vals)), 4), round(float(np.std(vals)), 4)
        mae, rmse = mean_std("mae_db"), mean_std("rmse_db")
        band, succ = mean_std("band_mae_db"), mean_std("success_rate_1db")
        rows.append({
            "encoder": encoder,
            "mode": mode,
            "n_seeds": len(group),
            "mae_db_mean": mae[0], "mae_db_std": mae[1],
            "rmse_db_mean": rmse[0], "rmse_db_std": rmse[1],
            "band_mae_db_mean": band[0],
            "success_rate_1db_mean": succ[0],
            "trainable_params": int(group[0]["trainable_params"]),
            "train_sec_mean": round(float(np.mean([g["train_sec"] for g in group])), 1),
        })
    return rows


ALL_RUNS, SHARD_DIRS = load_all_runs()
SUMMARY = aggregate(ALL_RUNS)

RESULTS = [r for r in ALL_RUNS if r["mode"] == "bimodal"]
LEADERBOARD = sorted(RESULTS, key=lambda r: r["validation"]["mae_db"])

print(f"loaded {len(ALL_RUNS)} run dari shard: {SHARD_DIRS or '(in-session)'}")
if SUMMARY:
    import csv
    csv_path = OUT_DIR / "audio_encoder_results.csv"
    with csv_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(SUMMARY[0].keys()))
        writer.writeheader()
        writer.writerows(SUMMARY)
    print("wrote", csv_path)
    print(f"{'encoder':12s} {'mode':11s} {'MAE±std':>16s} {'RMSE':>8s} {'band':>8s} {'succ<1dB':>9s} {'params':>10s}")
    for r in SUMMARY:
        print(f"{r['encoder']:12s} {r['mode']:11s} {r['mae_db_mean']:8.4f}±{r['mae_db_std']:<7.4f} "
              f"{r['rmse_db_mean']:8.4f} {r['band_mae_db_mean']:8.4f} {r['success_rate_1db_mean']:9.2f} {r['trainable_params']:10d}")
    if LEADERBOARD:
        print("\nBest bimodal encoder:", LEADERBOARD[0]["encoder"], "MAE =", LEADERBOARD[0]["validation"]["mae_db"])
else:
    print("No runs yet. Run Cell 9 with RUN_EXPERIMENT = True (masing-masing anggota).")

In [11]:
# ----------------------------------------------------------------------------- Cell 11: figure (bimodal MAE by encoder)
if SUMMARY:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    bimodal = [r for r in SUMMARY if r["mode"] == "bimodal"]
    if bimodal:
        labels = [f"{r['encoder']}\n{r['mode']}" for r in SUMMARY]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(labels, [r["mae_db_mean"] for r in SUMMARY], yerr=[r["mae_db_std"] for r in SUMMARY], capsize=3)
        ax.set_ylabel("Validation MAE (dB)")
        ax.set_title("Audio encoder comparison (fine-tuned IndoBERT, M3 head)")
        fig.tight_layout()
        fig_path = OUT_DIR / "audio_encoder_mae.png"
        fig.savefig(fig_path, dpi=150)
        print("wrote", fig_path)
        plt.close(fig)
else:
    print("nothing to plot yet")

nothing to plot yet


## 10. Parity / sanity checks

Memastikan perbandingan benar-benar apple-to-apple dan **tidak ada encoder yang frozen**.

In [ ]:
# ----------------------------------------------------------------------------- Cell 12: parity checks
FIXED_KEYS = ["text_model", "d_model", "batch_size", "grad_accum_steps", "epochs", "patience",
              "lr_head", "lr_encoder", "weight_decay", "grad_clip", "seeds"]
for key in FIXED_KEYS:
    assert key in CONFIG, f"missing fixed knob: {key}"
assert set(CONFIG["modes"]) == {"bimodal", "audio_only", "text_only"}
assert CONFIG["batch_size"] * CONFIG["grad_accum_steps"] == effective_batch, "effective batch harus sama (16)"
assert MEMBER in SHARDS

# Pembagian shard: 4 encoder x {bimodal, audio_only} + tepat satu text_only.
coverage = collections.defaultdict(set)
for member, shard in SHARDS.items():
    for enc in shard["encoders"]:
        for mode in shard["modes"]:
            coverage[(enc, mode)].add(member)
expected = {(e, m) for e in CONFIG["encoders"] for m in ("bimodal", "audio_only")}
missing = expected - set(coverage)
assert not missing, f"shard tidak menutup: {sorted(missing)}"
text_only_members = [m for m, s in SHARDS.items() if "text_only" in s["modes"]]
assert len(text_only_members) == 1, f"text_only harus tepat satu anggota, bukan {text_only_members}"
total_runs = sum(len(s["encoders"]) * len(s["modes"]) for s in SHARDS.values()) * len(CONFIG["seeds"])
assert total_runs == 27, f"total run harus 27, bukan {total_runs}"
print(f"shard coverage OK | 27 run | text_only oleh member {text_only_members[0]}")

# Nothing frozen: every parameter of a probe model must require grad.
probe = EncoderProbeModel("cnn_scratch", "bimodal", CONFIG)
frozen = [name for name, p in probe.named_parameters() if not p.requires_grad]
assert not frozen, f"frozen params found: {frozen}"
assert all(p.requires_grad for p in probe.text_branch.model.parameters())

# text_only head tidak boleh punya encoder (agar anggota 2 tidak mengunduh encoder berat).
assert EncoderProbeModel("ast", "text_only", CONFIG).encoder is None

# Runs must come from the agreed grid/protocol.
if ALL_RUNS:
    assert {r["seed"] for r in ALL_RUNS} <= set(CONFIG["seeds"])
    assert {r["encoder"] for r in ALL_RUNS} <= set(CONFIG["encoders"])
    assert {r["mode"] for r in ALL_RUNS} <= set(CONFIG["modes"])
    print(f"runs: {len(ALL_RUNS)} | {len({r['encoder'] for r in ALL_RUNS})} enc × "
          f"{len({r['mode'] for r in ALL_RUNS})} mode × {len({r['seed'] for r in ALL_RUNS})} seed")
print("parity checks passed.")

## 11. Kesimpulan & handoff ke step 2

1. Pilih encoder pemenang dari tabel **bimodal** (val MAE terendah, mean ± std antar seed).
2. Laporkan juga ablasi `audio_only` vs `text_only` sebagai kontrol unimodal, dan band MAE untuk
   keseimbangan spektral.
3. Bawa encoder pemenang ke **step 2**: sweep seluruh jenis fusion (late concat, cross-attention, FiLM, …)
   memakai encoder yang sama, sehingga perbandingan fusion juga apple-to-apple.

**Keterbatasan yang harus ditulis di laporan:**
- MERT diberi input 16 kHz (aslinya 24 kHz) — domain shift.
- BEATs/MERT/AST pretrained pada dataset berbeda (AudioSet vs musik) — perbandingan pretraining tidak netral.
- Foto jumlah parameter berbeda antar encoder; yang disamakan adalah protokol, bukan ukuran model.